# Neuron Flow 0.2.0 — ใช้งานบน Google Colab

รันใน runtime ใหม่: **1 ติดตั้ง ZIP → 2 อัปโหลด CSV → 3 ดูรายงาน → 4 ทำต่อ → 5 ตรวจชุดทดสอบ → 6 ดาวน์โหลด backup**
ไม่ต้องแก้โค้ดภายใน ไม่ต้องใช้ API key เซลล์เรียก CLI ผ่าน subprocess; Agent อยู่ในโมดูลของโปรเจกต์

ผลตรวจในเครื่อง: ฐานเดิม 38 กรณีและชุดใหม่ 65 กรณีผ่านแยกกัน รวมชุดอ้างอิง 100 เอกสาร (80 ถูก/20 ผิด)
ยังไม่ได้ตรวจบน Colab จริงก่อนจัดทำ Notebook นี้ การเปิดไฟล์หรือผ่าน syntax check ไม่ทำให้ขั้น 6 ผ่านเอง
ผู้ใช้อนุมัติให้ปิดรุ่นและเผยแพร่ 0.2.0 พร้อมข้อจำกัดว่า Colab ยังไม่ผ่านการตรวจจริง
หลักฐานการอนุมัติอยู่ใน RELEASE_APPROVAL.json; การอนุมัตินี้ไม่ได้หมายความว่าขั้น 6 ผ่านแล้ว

Colab เป็นพื้นที่ชั่วคราว ควรดาวน์โหลด backup เมื่อจบ session หากมี backup จากครั้งก่อน ให้รันข้อ 1
แล้วข้ามไป **เซลล์คืน backup ท้าย Notebook** จากนั้นรันข้อ 4 ได้โดยไม่อัปโหลด CSV ซ้ำ


## 1. อัปโหลด ZIP รุ่น 0.2.0 และตรวจ SHA-256

เลือก **neuron_flow_v0.2.0.zip เพียงไฟล์เดียว** ระบบตรวจเส้นทาง ZIP, ขนาด และ SHA-256
ทุกไฟล์ก่อนเรียกโค้ดโปรเจกต์ ใช้ Python standard library; ไม่ต้องติดตั้ง dependency เพิ่ม
รันเซลล์นี้ครั้งเดียวต่อ runtime เพื่อรักษาไฟล์และผลตรวจเดิม


In [ ]:
from google.colab import files
from pathlib import Path, PurePosixPath
from datetime import datetime, timezone
from contextlib import closing
import ast
import csv
import hashlib
import io
import json
import os
import platform
import shutil
import sqlite3
import stat
import subprocess
import sys
import tempfile
import zipfile

if sys.version_info < (3, 10):
    raise RuntimeError('ต้องใช้ Python 3.10 ขึ้นไป')

PROJECT = Path('/content/neuron_flow_project/neuron_flow')
WORKSPACE = Path('/content/neuron_flow_workspace')
SESSION_PATH = Path('/content/neuron_flow_session.json')
OBSERVATIONS = {'runtime': 'Google Colab', 'python': sys.version, 'platform': platform.platform(),
                'started_at_utc': datetime.now(timezone.utc).isoformat(),
                'download_received_confirmed': False}

def safe_unpack(data, destination, allowed_roots):
    if len(data) > 64 * 1024 * 1024:
        raise ValueError('ZIP ต้องไม่เกิน 64 MiB')
    destination = Path(destination)
    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(io.BytesIO(data)) as archive:
        entries = archive.infolist()
        if len(entries) > 5000 or sum(item.file_size for item in entries) > 256 * 1024 * 1024:
            raise ValueError('ZIP มีจำนวนไฟล์หรือขนาดหลังแตกเกินขอบเขต')
        seen = set()
        for item in entries:
            relative = PurePosixPath(item.filename)
            mode = (item.external_attr >> 16) & 0xffff
            if (not relative.parts or relative.is_absolute() or '..' in relative.parts
                    or '\\' in item.filename or any(':' in part for part in relative.parts)
                    or relative.parts[0] not in allowed_roots
                    or stat.S_IFMT(mode) not in (0, stat.S_IFREG, stat.S_IFDIR)
                    or item.flag_bits & 1 or item.file_size > 64 * 1024 * 1024):
                raise ValueError('พบเส้นทางหรือชนิดไฟล์ที่ไม่รองรับใน ZIP')
            target = (destination / item.filename).resolve()
            if not target.is_relative_to(destination.resolve()) or target in seen:
                raise ValueError('ZIP มีเส้นทางซ้ำหรืออยู่นอกโฟลเดอร์ปลายทาง')
            seen.add(target)
        archive.extractall(destination)

def verify_sums(root, manifest_name='SHA256SUMS.txt'):
    root = Path(root)
    checked = set()
    for line in (root / manifest_name).read_text(encoding='utf-8').splitlines():
        digest, separator, name = line.partition('  ')
        relative = PurePosixPath(name)
        if (not separator or len(digest) != 64 or any(c not in '0123456789abcdef' for c in digest)
                or not relative.parts or relative.is_absolute() or '..' in relative.parts
                or '\\' in name or name == manifest_name or name in checked):
            raise ValueError('รูปแบบ SHA-256 manifest ไม่ถูกต้อง')
        path = root / name
        if (not path.resolve().is_relative_to(root.resolve()) or not path.is_file()
                or path.is_symlink() or hashlib.sha256(path.read_bytes()).hexdigest() != digest):
            raise ValueError('SHA-256 ไม่ตรง: ' + name)
        checked.add(name)
    actual = {p.relative_to(root).as_posix() for p in root.rglob('*')
              if p.is_file() and p.relative_to(root).as_posix() != manifest_name}
    if not checked or checked != actual:
        raise ValueError('Manifest ไม่ครอบคลุมไฟล์ทั้งหมด')
    return len(checked)

if PROJECT.exists():
    raise RuntimeError('มีโปรเจกต์ใน runtime นี้แล้ว ใช้เซลล์ถัดไป หรือเริ่ม runtime ใหม่ก่อนติดตั้ง ZIP')
uploaded = files.upload()
if len(uploaded) != 1 or not next(iter(uploaded)).lower().endswith('.zip'):
    raise ValueError('เลือก ZIP ของโปรเจกต์เพียงไฟล์เดียว')
safe_unpack(next(iter(uploaded.values())), PROJECT.parent, {'neuron_flow'})
verified_files = verify_sums(PROJECT)
required = ['neuron_flow/__main__.py', 'neuron_flow/batch.py', 'neuron_flow/reporting.py',
            'tools/verify_v020.py', 'acceptance/fixture_lock.json']
if not all((PROJECT / name).is_file() for name in required):
    raise ValueError('ชุดโปรเจกต์ไม่ครบ')
version = subprocess.check_output([sys.executable, '-c',
    'from neuron_flow import __version__; print(__version__)'], cwd=PROJECT, text=True).strip()
if version != '0.2.0':
    raise ValueError('ZIP ต้องเป็นรุ่น 0.2.0')
PACKAGE_MANIFEST_SHA = hashlib.sha256((PROJECT / 'SHA256SUMS.txt').read_bytes()).hexdigest()
OBSERVATIONS.update(project_version=version, project_files_verified=verified_files,
                    package_manifest_sha256=PACKAGE_MANIFEST_SHA)

def run_cli(*arguments):
    completed = subprocess.run([sys.executable, '-m', 'neuron_flow', *map(str, arguments)],
                               cwd=PROJECT, capture_output=True, text=True, timeout=300)
    if completed.stderr:
        print(completed.stderr)
    if completed.returncode not in (0, 1):
        print(completed.stdout)
        completed.check_returncode()
    result = json.loads(completed.stdout)
    print('exit code:', completed.returncode, 'สถานะ:', result.get('batch_status', result.get('status')))
    print('จำนวนเอกสาร:', result.get('counts', {}), 'ยอดที่รับรอง:', result.get('certified_total'))
    if completed.returncode == 1:
        print('พบ ERROR หรือส่งมอบยังไม่สำเร็จ โปรดตรวจผลรายเอกสารและ receipt')
    return result

print('พร้อมใช้:', PROJECT, '| Python', platform.python_version(), '| ตรวจ SHA-256 แล้ว', verified_files, 'ไฟล์')


## 2. อัปโหลด CSV และประมวลผล

เลือก CSV **หนึ่งไฟล์** UTF-8 หรือ UTF-8 BOM ไม่เกิน 16 MiB ใช้ชื่อช่อง
`document_id,item_id,description,quantity,unit_price,declared_total,currency`
หากต้องการตัวอย่าง ไฟล์ `examples/purchases.csv` และ `examples/purchases_invalid.csv` อยู่ใน ZIP
รหัส batch สร้างจากเนื้อหา/รุ่นโดยอัตโนมัติ ข้อมูลผิดจะเป็น INVALID แยกใบ ส่วน ERROR หมายถึงระบบยืนยันผลไม่ได้
ยอดที่รับรองรวมเฉพาะ VALID; แถวไม่มีรหัสเก็บใน ungrouped_rows

เมื่อคืน backup ให้ข้ามเซลล์นี้และไปข้อ 4


In [ ]:
uploaded_csv = files.upload()
if len(uploaded_csv) != 1 or not next(iter(uploaded_csv)).lower().endswith('.csv'):
    raise ValueError('เลือก CSV เพียงไฟล์เดียว')
csv_bytes = next(iter(uploaded_csv.values()))
if len(csv_bytes) > 16 * 1024 * 1024:
    raise ValueError('CSV ต้องไม่เกิน 16 MiB')
upload_directory = Path('/content/neuron_flow_uploads')
upload_directory.mkdir(exist_ok=True)
source_path = upload_directory / 'input.csv'
source_path.write_bytes(csv_bytes)
RESULT = run_cli('csv', source_path, '--workspace', WORKSPACE)
BATCH_ID = RESULT['batch_id']
SESSION_PATH.write_text(json.dumps({'batch_id': BATCH_ID, 'workspace': str(WORKSPACE),
    'project_version': version, 'package_manifest_sha256': PACKAGE_MANIFEST_SHA},
    ensure_ascii=False, indent=2), encoding='utf-8')
OBSERVATIONS.update(csv_uploaded=True, csv_sha256=hashlib.sha256(csv_bytes).hexdigest(),
                    batch_id=BATCH_ID, cli_counts=RESULT.get('counts'), cli_executed=True)
print('batch_id:', BATCH_ID)
print('แถวที่จัดกลุ่มไม่ได้:', RESULT.get('ungrouped_rows'))
print(json.dumps(RESULT.get('receipt', {}), ensure_ascii=False, indent=2))


## 3. ดูตัวอย่างรายงาน

ผลรายใบมี VALID / INVALID / ERROR และ engine_status แยกกัน การส่งรายงานสำเร็จยังต้องแสดง ERROR ทุกใบ
source_rows นับบรรทัดจริงเริ่มต้น record; ข้อความ CSV หลายบรรทัดอาจทำให้เลขแถวไม่เรียงติดกัน


In [ ]:
session = json.loads(SESSION_PATH.read_text(encoding='utf-8'))
BATCH_ID = session['batch_id']
STATUS = run_cli('batch-status', BATCH_ID, '--workspace', WORKSPACE)
artifact_info = (STATUS.get('receipt') or {}).get('artifacts', {})
if not artifact_info:
    raise RuntimeError('ยังไม่มีรายงานครบชุด ให้ตรวจสถานะและใช้เซลล์ทำต่อ')
summary_path = Path(artifact_info['summary.csv']['path'])
with summary_path.open(encoding='utf-8', newline='') as handle:
    preview = list(csv.DictReader(handle))[:10]
print(json.dumps(preview, ensure_ascii=False, indent=2))
details = json.loads(Path(artifact_info['details.json']['path']).read_bytes())
print('ข้อมูลระดับชุด:', {key: details[key] for key in ('counts', 'certified_total', 'source_sha256')})
print('ungrouped_rows:', len(details['ungrouped_rows']))
OBSERVATIONS['report_previewed'] = True


## 4. ทำต่อโดยไม่อัปโหลดต้นทางซ้ำ

ใช้ workspace เดิม ระบบเก็บ CSV ใน SQLite แล้ว ใช้ผลสำเร็จเดิมและไม่สร้างรายงานเดิมซ้ำ
ค่าเริ่มต้นลองรวมไม่เกิน 2 ครั้ง; ใบที่หมดงบยังเป็น ERROR งานสำเร็จไม่ถูกลองใหม่
หากคืน backup ใน runtime ใหม่แล้ว รันเซลล์นี้ได้ทันที


In [ ]:
session = json.loads(SESSION_PATH.read_text(encoding='utf-8'))
if session['workspace'] != str(WORKSPACE) or not (WORKSPACE / 'state.sqlite3').is_file():
    raise RuntimeError('ไม่พบ workspace เดิม ให้คืน backup ก่อน')
BATCH_ID = session['batch_id']
RESULT = run_cli('batch-resume', BATCH_ID, '--workspace', WORKSPACE)
OBSERVATIONS.update(resume_executed=True, resume_counts=RESULT.get('counts'), batch_id=BATCH_ID)
print(json.dumps(RESULT.get('receipt', {}), ensure_ascii=False, indent=2))


## 5. ตรวจฐานเดิมและชุดใหม่ พร้อมเก็บหลักฐาน

คำสั่งนี้ตรวจฐานเดิม 38 กรณีแยกจากชุดใหม่ 65 กรณี รัน 100 เอกสารเทียบคำตอบที่ล็อก
และกรณี Agent ค้าง/ตาย controller ถูกหยุดจริง รวมถึงการส่งมอบและเปิดต่อ อาจใช้เวลามากกว่าตัวอย่าง
ผลแสดง environment และข้อจำกัดตามที่ตรวจจริง `release_ready` อิงผลตรวจในเครื่องและ
การอนุมัติใน RELEASE_APPROVAL.json ส่วน `colab_acceptance_complete` ยังคงเป็น false
จนตรวจ Colab จริงครบ หลักฐานอัปโหลดถึงดาวน์โหลดยังอยู่ใน `remaining_acceptance`


In [ ]:
checked = subprocess.run([sys.executable, 'tools/verify_v020.py'], cwd=PROJECT,
                         capture_output=True, text=True, timeout=300)
print(checked.stdout)
if checked.stderr:
    print(checked.stderr)
checked.check_returncode()
verification = json.loads((PROJECT / 'verification/v0_2_0/test_results.json').read_text(encoding='utf-8'))
print(json.dumps({key: verification[key] for key in
    ('version', 'baseline_38_passed', 'new_test_cases', 'failures', 'errors', 'skipped',
     'local_successful', 'colab', 'release_ready', 'remaining_acceptance')}, ensure_ascii=False, indent=2))
OBSERVATIONS.update(verifier_executed=True, verifier_local_successful=verification['local_successful'],
                    observed_colab=verification['colab'])


## 6. ดาวน์โหลด workspace และหลักฐาน

ต้องรอให้คำสั่งก่อนหน้าจบก่อนดาวน์โหลด เซลล์สำรอง SQLite ด้วย backup API แล้วรวมรายงาน
verification และ session metadata พร้อม SHA-256 เก็บ ZIP นี้ไว้ เพราะ Colab ลบไฟล์เมื่อ session สิ้นสุด
ตรวจว่าเบราว์เซอร์ได้รับไฟล์จริง การเรียก `files.download` อย่างเดียวยังไม่ยืนยันว่าดาวน์โหลดถึงเครื่องแล้ว

การคืน backup ต้องใช้โปรเจกต์ ZIP เดิมและ path เดิม `/content/neuron_flow_workspace`
เพราะ receipt ใน SQLite เก็บ absolute path ไฟล์หลักฐาน session จะไม่อ้างว่าดาวน์โหลดสำเร็จโดยอัตโนมัติ


In [ ]:
if not (WORKSPACE / 'state.sqlite3').is_file() or not SESSION_PATH.is_file():
    raise RuntimeError('ยังไม่มี workspace/session ให้สำรอง')
OBSERVATIONS['backup_prepared_at_utc'] = datetime.now(timezone.utc).isoformat()
observation_path = PROJECT / 'verification/v0_2_0/colab_notebook_session.json'
observation_path.parent.mkdir(parents=True, exist_ok=True)
observation_path.write_text(json.dumps(OBSERVATIONS, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
stage = Path(tempfile.mkdtemp(prefix='neuron_flow_backup_', dir='/content'))
shutil.copytree(WORKSPACE, stage / 'workspace', ignore=shutil.ignore_patterns(
    'state.sqlite3', 'state.sqlite3-wal', 'state.sqlite3-shm', '*.tmp'))
with closing(sqlite3.connect('file:' + str(WORKSPACE / 'state.sqlite3') + '?mode=ro', uri=True)) as original_db:
    with closing(sqlite3.connect(stage / 'workspace/state.sqlite3')) as backup_db:
        original_db.backup(backup_db)
shutil.copytree(PROJECT / 'verification', stage / 'verification')
shutil.copy2(SESSION_PATH, stage / 'session.json')
manifest = []
for path in sorted(stage.rglob('*')):
    if path.is_symlink():
        raise ValueError('พบ symlink ใน workspace ที่จะสำรอง')
    if path.is_file():
        manifest.append(hashlib.sha256(path.read_bytes()).hexdigest() + '  ' + path.relative_to(stage).as_posix())
(stage / 'backup_SHA256SUMS.txt').write_text('\n'.join(manifest) + '\n', encoding='utf-8')
verify_sums(stage, 'backup_SHA256SUMS.txt')
backup_path = Path('/content/neuron_flow_workspace_backup.zip')
with zipfile.ZipFile(backup_path, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(stage.rglob('*')):
        if path.is_file():
            archive.write(path, path.relative_to(stage).as_posix())
print('backup ZIP SHA-256:', hashlib.sha256(backup_path.read_bytes()).hexdigest())
print('สำรองแล้ว:', backup_path, '| bytes:', backup_path.stat().st_size)
files.download(str(backup_path))


## ทางเลือก: คืน backup ใน Colab runtime ใหม่

สำหรับงานเดิมเท่านั้น: รันข้อ 1 ด้วย ZIP โปรเจกต์เดิม แล้วรันเซลล์นี้โดยเลือก
**neuron_flow_workspace_backup.zip เพียงไฟล์เดียว** จากนั้นไปข้อ 4 เพื่อทำต่อ
ไม่ต้องรันเซลล์อัปโหลด CSV ใหม่ ระบบจะตรวจ hash ทุกไฟล์และปฏิเสธการเขียนทับ workspace ที่มีข้อมูล


In [ ]:
if WORKSPACE.exists() and any(WORKSPACE.iterdir()):
    raise RuntimeError('workspace มีข้อมูลอยู่แล้ว จึงไม่คืน backup ทับ')
restored_upload = files.upload()
if len(restored_upload) != 1 or not next(iter(restored_upload)).lower().endswith('.zip'):
    raise ValueError('เลือก ZIP backup เพียงไฟล์เดียว')
restore_stage = Path(tempfile.mkdtemp(prefix='neuron_flow_restore_', dir='/content'))
safe_unpack(next(iter(restored_upload.values())), restore_stage,
            {'workspace', 'verification', 'session.json', 'backup_SHA256SUMS.txt'})
verify_sums(restore_stage, 'backup_SHA256SUMS.txt')
restored_session = json.loads((restore_stage / 'session.json').read_text(encoding='utf-8'))
if (restored_session['workspace'] != str(WORKSPACE)
        or restored_session['project_version'] != '0.2.0'
        or restored_session['package_manifest_sha256'] != PACKAGE_MANIFEST_SHA):
    raise ValueError('backup ต้องใช้ path และ ZIP โปรเจกต์เดิม')
if not (restore_stage / 'workspace/state.sqlite3').is_file():
    raise ValueError('backup ไม่มี SQLite workspace')
shutil.copytree(restore_stage / 'workspace', WORKSPACE, dirs_exist_ok=True)
shutil.copytree(restore_stage / 'verification', PROJECT / 'verification', dirs_exist_ok=True)
shutil.copy2(restore_stage / 'session.json', SESSION_PATH)
BATCH_ID = restored_session['batch_id']
OBSERVATIONS.update(backup_restored=True, batch_id=BATCH_ID)
print('คืน workspace แล้ว:', WORKSPACE, '| batch_id:', BATCH_ID)
print('ไปข้อ 4 เพื่อทำต่อโดยไม่อัปโหลด CSV ซ้ำ')
